# Horovod

A comprehensive guide to Horovod for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

Horovod is an open-source framework for **distributed deep learning** originally developed at Uber. It provides a unified, high-performance way to run data-parallel training across multiple GPUs and machines for frameworks like PyTorch and TensorFlow.

### What is it?

At a high level, **Horovod**:

- Implements efficient **all-reduce**–based gradient synchronization (often ring all-reduce).
- Integrates with popular deep learning frameworks via thin adapters.
- Supports a variety of communication backends (MPI, Gloo, NCCL).

### Why use it?

Key benefits of using Horovod:

- **Framework-agnostic**: Works with PyTorch, TensorFlow, Keras, MXNet, and more.
- **Scales well**: Designed for large-scale training on multi-GPU and multi-node clusters.
- **Simple integration**: Wrap existing training code with a few Horovod calls instead of rewriting from scratch.

### When to use it?

Horovod is particularly useful when:

- You need a **unified distributed training layer** across multiple frameworks.
- You already use **MPI-based** or HPC-style clusters.
- You want well-tested **ring all-reduce**–style gradient synchronization at scale.

## Key Features

### Core Capabilities of Horovod

| Feature | Description | Benefit |
|--------|-------------|---------|
| **Multi-framework support** | Adapters for PyTorch, TensorFlow, Keras, MXNet, etc. | Use one distributed training approach across different model stacks. |
| **Ring all-reduce** | Efficient gradient synchronization algorithm. | Good scaling characteristics on many-node GPU clusters. |
| **MPI / Gloo / NCCL backends** | Can run on top of MPI, Gloo, and NCCL. | Flexible deployment in HPC, cloud, and on-prem clusters. |
| **Elastic training** | Support for elastic (resizable) training jobs. | Better resiliency and resource utilization. |
| **Tensor Fusion** | Combines many small gradient tensors into larger buffers. | Reduces communication overhead and improves throughput. |

## Architecture Overview

Horovod typically runs **one process per GPU** and uses an all-reduce–based pattern to synchronize gradients.

```text
+-----------+    +-----------+    +-----------+
| Rank 0    |    | Rank 1    |    | Rank 2    |
| GPU 0     |    | GPU 1     |    | GPU 2     |
+-----+-----+    +-----+-----+    +-----+-----+
      |                |                |
      +------ ring all-reduce (NCCL/MPI/Gloo) ----+
```

### Key components

1. **Horovod processes (ranks)**  
   - One process per GPU is common.  
   - Each process runs a copy of the model and computes gradients on its data shard.

2. **All-reduce communication**  
   - Gradients are averaged across ranks using ring all-reduce or similar algorithms.

3. **Framework adapters**  
   - E.g., `horovod.torch.DistributedOptimizer` wraps a PyTorch optimizer.  
   - E.g., `hvd.broadcast_parameters` syncs initial model weights across ranks.

4. **Launch mechanism**  
   - Can be launched with `horovodrun`, `mpirun`, or through job schedulers (Slurm, Kubernetes, etc.).

## Installation

### Prerequisites

- Python 3.8+.
- A supported deep learning framework (e.g., PyTorch or TensorFlow).
- MPI, Gloo, or another supported communication backend, plus CUDA drivers for GPU training.

### Install Horovod

For PyTorch, a typical installation looks like:

```bash
pip install "horovod[pytorch]"
```

For TensorFlow, you might use:

```bash
pip install "horovod[tensorflow]"
```

Always consult the official Horovod docs for up-to-date installation instructions and GPU/MPI compatibility notes.

In [ ]:
# Quick install helper for notebooks (uncomment to run)
# !pip install "horovod[pytorch]" torch torchvision

## Basic Usage

### Quick start: PyTorch + Horovod

The typical pattern is:

1. Initialize Horovod with `hvd.init()`.
2. Pin each process to a specific GPU via `hvd.local_rank()`.
3. Wrap your optimizer with `hvd.DistributedOptimizer`.
4. Broadcast initial parameters and optimizer state from rank 0.

This minimal example illustrates a simple training loop (conceptually saved as `train_horovod.py` and launched with `horovodrun` or `mpirun`).

In [ ]:
# Minimal Horovod + PyTorch example (intended for a script)

import torch
import torch.nn as nn
import torch.optim as optim

import horovod.torch as hvd


# 1. Initialize Horovod
hvd.init()

# 2. Pin GPU to local rank
if torch.cuda.is_available():
    torch.cuda.set_device(hvd.local_rank())
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


class ToyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)


model = ToyModel().to(device)

# 3. Scale learning rate by number of workers
optimizer = optim.SGD(model.parameters(), lr=0.01 * hvd.size())

# 4. Wrap optimizer with Horovod's DistributedOptimizer
optimizer = hvd.DistributedOptimizer(
    optimizer,
    named_parameters=model.named_parameters(),
)

# 5. Broadcast parameters & optimizer state from rank 0
hvd.broadcast_parameters(model.state_dict(), root_rank=0)
hvd.broadcast_optimizer_state(optimizer, root_rank=0)


for step in range(5):
    x = torch.randn(32, 10, device=device)
    y = torch.randn(32, 1, device=device)

    optimizer.zero_grad()
    outputs = model(x)
    loss = ((outputs - y) ** 2).mean()
    loss.backward()
    optimizer.step()

    if hvd.rank() == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

# Launch with something like:
# horovodrun -np 4 python train_horovod.py
# or
# mpirun -np 4 python train_horovod.py

## Advanced Features

- **Elastic training**: Adjust the number of workers at runtime without restarting from scratch.  
- **Horovod on Spark**: Integrate with Apache Spark for unified data processing + training pipelines.  
- **Mixed precision & gradient compression**: Combine FP16 training and gradient compression to reduce communication volume.  
- **Keras, TensorFlow, MXNet support**: Similar APIs for each framework so you can reuse distributed concepts across stacks.

In [ ]:
# Sketch: enabling gradient compression (conceptual)

# optimizer = hvd.DistributedOptimizer(
#     optimizer,
#     named_parameters=model.named_parameters(),
#     compression=hvd.Compression.fp16,
# )

print("Horovod can optionally compress gradients to reduce bandwidth.")

## Use Cases

- Training image classification models across many GPUs using PyTorch or TensorFlow.  
- Large-scale NLP model training where ring all-reduce works well on your cluster network.  
- Environments where MPI is already standard, such as HPC clusters.  
- Spark-based workflows where you want to unify ETL and model training.

## Best Practices

1. **Validate single-GPU training first** before introducing Horovod.  
2. **Use one process per GPU** as the common deployment model.  
3. **Tune `fusion_buffer` size** and other Horovod-specific parameters for your network.  
4. **Align batch size and learning rate** when scaling to many workers (e.g., linear scaling rules).  
5. **Log only from rank 0** or a small set of ranks to avoid cluttered logs.

## Common Pitfalls

1. **MPI or NCCL misconfiguration**  
   - Symptom: Hangs or cryptic NCCL/MPI errors.  
   - Fix: Validate a simple MPI or NCCL-only script; ensure environment variables and networking are correct.

2. **Not scaling the learning rate**  
   - Symptom: Training diverges or converges slowly after increasing worker count.  
   - Fix: Adjust learning rate according to the effective global batch size.

3. **Improper GPU binding**  
   - Symptom: Multiple processes contend for the same GPU.  
   - Fix: Always use `hvd.local_rank()` to pick the correct GPU per process.

4. **Incorrect launch commands**  
   - Symptom: Unexpected world size or ranks.  
   - Fix: Ensure `-np` matches the number of processes and that hosts are correctly specified.

## Performance Optimization

- **Use NCCL for GPU all-reduce** when possible for the best performance on NVIDIA GPUs.  
- **Experiment with fusion and cycle times** (Horovod-specific settings) to optimize communication.  
- **Profile communication vs. computation** to identify bottlenecks.  
- **Use mixed precision and gradient compression** to reduce communication volume and memory usage.

In [ ]:
# Placeholder for custom benchmarking logic

print("Use timers and throughput metrics to benchmark Horovod training.")

## Production Deployment

- **HPC / MPI clusters**: Launch Horovod jobs with `mpirun` or scheduler-integrated tools (e.g., Slurm integration).  
- **Kubernetes**: Use operators (e.g., Kubeflow Training Operator) or custom controllers to manage multi-replica Horovod jobs.  
- **Cloud managed services**: Wrap Horovod scripts into batch jobs or containerized tasks, ensuring networking and hostfile configuration is handled by the platform.

## Monitoring and Observability

- Track **step time**, **throughput (samples/sec)**, and **GPU utilization** across all ranks.  
- Collect logs from all workers, but aggregate or focus on rank 0 for summary metrics.  
- Use cluster monitoring tools (Prometheus, Grafana, cloud provider dashboards) to watch network and GPU health.  
- Include Horovod-related metrics (e.g., time spent in all-reduce) where available.

## Troubleshooting

- **Hangs at startup**: Verify all hosts can reach each other and that MPI is correctly installed and configured.  
- **NCCL timeout or errors**: Check driver/CUDA versions and network settings; test NCCL with smaller jobs.  
- **Diverging loss after scaling up**: Revisit learning rate scaling and gradient accumulation.  
- **Inconsistent performance across runs**: Monitor cluster load, ensure no node is oversubscribed or has hardware issues.

## Comparison with Alternatives

| Aspect | Horovod | PyTorch DDP / TF native | DeepSpeed / Ray Train |
|--------|---------|-------------------------|------------------------|
| Framework support | Multi-framework | Framework-specific | Varies (often PyTorch-first) |
| Communication backend | MPI / Gloo / NCCL | NCCL / Gloo | Uses underlying frameworks |
| Large model features | Less focus on ZeRO-style sharding | FSDP / TF strategies | ZeRO, offload, advanced strategies |
| Integration style | External library wrapping optimizers | Native APIs | Higher-level orchestration |

Use Horovod when you want **one library to scale multiple frameworks** and already rely on MPI-style infrastructure.

## Resources

- Horovod homepage: https://horovod.ai/  
- Getting started guide: https://horovod.ai/getting-started/  
- GitHub repository: https://github.com/horovod/horovod

Look for framework-specific examples (PyTorch, TensorFlow, Keras) in the official docs and GitHub examples directory.